# Back up and restore the Azure ML workspace

The GPU workspace is expensive to keep idle, so it can be torn down when not in use for an extended period. A **full teardown also destroys the storage account** backing the datastore, so every datastore blob is lost. This notebook walks through the full cycle:

1. **Back up** all irreplaceable workspace contents to local disk (`scripts/aml_backup.sh`).
2. **Tear down** the workspace (external — your existing spin-down mechanism).
3. **Spin up** a fresh bare workspace (external — creates the workspace, `large_datastore`, and compute cluster).
4. **Restore** the environments and data into the new workspace (`scripts/aml_restore.sh`).

What is backed up: ground-truth transcriptions, `outputs/{checkpoints,extractions,eval}`, `consensus_data`, `test_data`, and the local registries / `config.env` / `azureml/*.yml` specs. The 660k raw source images and `hf_cache` are **excluded by default** — the images are re-derivable from the NMLA archive and the cache is re-downloaded automatically by jobs. Pass `--include-images` to capture the raw images too.

> **Environment.** Run this notebook in the `weather-doc-extractor` conda environment, authenticated to Azure (`az login`). Workspace coordinates come from `azureml/config.env`.


## Configuration

Locate the repository root, choose the local backup directory, and define a small helper to run the backup/restore shell scripts.

The backup directory is **required** and can be hundreds of gigabytes, so point `BACKUP_DIR` at a large-capacity location **outside the repository** (not the 6 GB local disk root). You can also set it once via `AML_BACKUP_DIR` in `azureml/config.env`.


In [2]:

import os
import subprocess
from pathlib import Path

# Locate the repository root (the folder containing `scripts/`).
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "scripts").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
SCRIPTS = REPO_ROOT / "scripts"

# Local backup bundle directory. REQUIRED and potentially hundreds of GB, so use
# a large-capacity path OUTSIDE the repository. Edit this for a real run.
BACKUP_DIR = Path("%s/llmdatarescue-Workspace-Backup" % os.getenv('SCRATCH'))  # Edit this to a real path for a real run.


def run(script, *args):
    """Run a repository shell script with bash and stream its output."""
    cmd = ["bash", str(SCRIPTS / script), *[str(a) for a in args]]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=REPO_ROOT)
    print(f"[exit code {result.returncode}]")
    return result.returncode


REPO_ROOT, BACKUP_DIR


(PosixPath('/home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO'),
 PosixPath('/data/scratch/philip.brohan/llmdatarescue-Workspace-Backup'))

## 0. Prune unregistered components

Before backing anything up, drop workspace clutter so the bundle only contains artifacts we actually track. The existing cleanup scripts delete checkpoints/models and extractions/eval that are **not** referenced in the registries, on both local disk and the Azure datastore:

- `scripts/cleanup_unregistered_checkpoints.py` — removes checkpoints absent from `outputs/model_registry.json`.
- `scripts/cleanup_unregistered_outputs.py` — removes extractions/eval absent from `outputs/{model,extraction}_registry.json`.

Both default to a **dry-run**. The cell below previews the deletions with `APPLY = False`; set `APPLY = True` to actually remove the unregistered components, then re-run before proceeding to the backup.

> **Note on memory.** These scripts print one line per blob, which for the extractions can be hundreds of thousands of lines. A notebook keeps *all* cell output in memory, so streaming that directly would exhaust this 6 GB machine. The cell therefore redirects each script's full output to a log file under `/var/tmp/aml_prune_logs/` and shows only the exit code, line count, and last few lines. Open the log files if you need the complete list.


In [4]:
import sys

# Set to True to actually delete unregistered components (default: preview only).
APPLY = True

# These scripts print one line per blob. In dry-run over extractions that can be
# hundreds of thousands of lines — enough to exhaust notebook/kernel memory if
# streamed into the cell. So we redirect all output to a log file (low memory)
# and show only a short summary here.
LOG_DIR = Path("/var/tmp/aml_prune_logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)


def run_py(script, *args, tail=25):
    """Run a repository Python script, logging output to disk (not the notebook).

    Only the exit code, total line count, and the last ``tail`` lines are printed,
    so a run that emits millions of lines cannot overwhelm the notebook.
    """
    log_path = LOG_DIR / f"{Path(script).stem}.log"
    cmd = [sys.executable, str(SCRIPTS / script), *[str(a) for a in args]]
    print("$", " ".join(cmd))
    with open(log_path, "w") as log:
        result = subprocess.run(cmd, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT)
    with open(log_path) as log:
        lines = log.readlines()
    print(f"[exit code {result.returncode}]  {len(lines)} output line(s) → {log_path}")
    if lines:
        print(f"--- last {min(tail, len(lines))} line(s) ---")
        print("".join(lines[-tail:]).rstrip())
    return result.returncode


extra = ["--apply"] if APPLY else []
print(">>> APPLYING DELETIONS.\n" if APPLY else ">>> DRY RUN — set APPLY = True to perform deletions.\n")

# Remove checkpoints/models not present in the model registry (local + Azure).
run_py("cleanup_unregistered_checkpoints.py", *extra)
print()
# Remove extractions/eval not present in the registries (local + Azure).
run_py("cleanup_unregistered_outputs.py", *extra)


>>> APPLYING DELETIONS.

$ /data/users/philip.brohan/conda/environments/weather-doc-extractor/bin/python /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/cleanup_unregistered_checkpoints.py --apply
[exit code 0]  27 output line(s) → /var/tmp/aml_prune_logs/cleanup_unregistered_checkpoints.log
--- last 25 line(s) ---

Local cleanup plan:
  local checkpoints found: 0
  local checkpoints delete: 0

Azure datastore context:
  datastore: large_datastore
  account:   sallmdatarescue02
  container: default
  outputs:   Daily_rainfall_sample/outputs

Azure cleanup plan:
  azure checkpoints found: 10
  azure checkpoints delete: 0
  azure stray blobs delete: 5

Execution mode: APPLY

[DELETE] azure stray blob: Daily_rainfall_sample/outputs/checkpoints/HuggingFaceTB--SmolVLM2-2.2B-Instruct-20260629-124643
[DELETE] azure stray blob: Daily_rainfall_sample/outputs/checkpoints/google--gemma-3-4b-it-20260629-124722
[DELETE] azure stray blob: Daily_rainfall_sample/outputs/checkpoints/g

0

## 1. Preview the backup (dry run)

A `--dry-run` prints the `az storage` commands that would run and confirms the workspace, datastore, and destination — without transferring anything. Use it to sanity-check the configuration before a real backup.


In [5]:
run("aml_backup.sh", "--dest", BACKUP_DIR, "--dry-run")


$ bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_backup.sh --dest /data/scratch/philip.brohan/llmdatarescue-Workspace-Backup --dry-run
Azure ML workspace backup
  workspace:  mlw-llmdatarescue-uksouth-01
  datastore:  large_datastore
  dest:       /data/scratch/philip.brohan/llmdatarescue-Workspace-Backup
  images:     excluded

[dry-run] Would resolve datastore 'large_datastore' in workspace 'mlw-llmdatarescue-uksouth-01'

[dry-run] az storage blob download-batch \
    --account-name <account> --auth-mode login \
    --source <container> --pattern 'Daily_rainfall_sample/transcriptions/*' \
    --destination <tmpdir> --max-connections 16 --overwrite true
    (then move <tmpdir>/Daily_rainfall_sample/transcriptions/ → /data/scratch/philip.brohan/llmdatarescue-Workspace-Backup/datastore/Daily_rainfall_sample/transcriptions)

[dry-run] az storage blob download-batch \
    --account-name <account> --auth-mode login \
    --source <container> --pattern 'Daily_rai

0

[dry-run] cp /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/model_registry.json → /data/scratch/philip.brohan/llmdatarescue-Workspace-Backup/metadata/outputs/model_registry.json
[dry-run] cp /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extraction_registry.json → /data/scratch/philip.brohan/llmdatarescue-Workspace-Backup/metadata/outputs/extraction_registry.json
[dry-run] cp /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/azureml/config.env → /data/scratch/philip.brohan/llmdatarescue-Workspace-Backup/metadata/azureml/config.env
[dry-run] cp /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/azureml/conda-a100.yml → /data/scratch/philip.brohan/llmdatarescue-Workspace-Backup/metadata/azureml/conda-a100.yml
[dry-run] cp /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/azureml/conda.yml → /data/scratch/philip.brohan/llmdatarescue-Workspace-Backup/metadata/azureml/conda.yml
[dry-run] cp /home/users/philip.brohan/Projects/Auto-

## 2. Run the backup

Pull the datastore blobs and copy the local metadata into `BACKUP_DIR`. The download is idempotent and resumable, so an interrupted run is resumed simply by re-running the cell. Add `--include-images` to also capture the 660k raw source images.


In [7]:
# Real backup. Add "--include-images" to also capture the raw source images.
run("aml_backup.sh", "--dest", BACKUP_DIR)


$ bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_backup.sh --dest /data/scratch/philip.brohan/llmdatarescue-Workspace-Backup
Azure ML workspace backup
  workspace:  mlw-llmdatarescue-uksouth-01
  datastore:  large_datastore
  dest:       /data/scratch/philip.brohan/llmdatarescue-Workspace-Backup
  images:     excluded

Resolving datastore 'large_datastore' in workspace 'mlw-llmdatarescue-uksouth-01'...


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Storage account: sallmdatarescue02  container: default

Backing up: Daily_rainfall_sample/transcriptions
        to: /data/scratch/philip.brohan/llmdatarescue-Workspace-Backup/datastore/Daily_rainfall_sample/transcriptions

Job 5863a28d-75ca-5d49-5240-3596b9a29248 has started
Log file is located at: /data/scratch/philip.brohan/azcopy/5863a28d-75ca-5d49-5240-3596b9a29248.log




Job 5863a28d-75ca-5d49-5240-3596b9a29248 summary
Elapsed Time (Minutes): 0.0334
Number of File Transfers: 322
Number of Folder Property Transfers: 0
Number of Symlink Transfers: 0
Total Number of Transfers: 322
Number of File Transfers Completed: 0
Number of Folder Transfers Completed: 0
Number of File Transfers Failed: 0
Number of Folder Transfers Failed: 0
Number of File Transfers Skipped: 322
Number of Folder Transfers Skipped: 0
Total Number of Bytes Transferred: 0
Final Job Status: CompletedWithSkipped

Done.

Backing up: Daily_rainfall_sample/outputs/checkpoints
        to: /data/scratch/philip.brohan/llmd

0

## 3. Inspect the bundle

Confirm the backup completed by reading `manifest.json` and listing what was captured. Keep this bundle safe until the new workspace is fully restored and verified.


In [8]:
import json

manifest = json.loads((BACKUP_DIR / "manifest.json").read_text())
print(f"Created:    {manifest['created_at']}")
print(f"Workspace:  {manifest['workspace']}")
print(f"Datastore:  {manifest['datastore_name']}")
print(f"Images:     {'included' if manifest['include_images'] else 'excluded'}")

print("\nDatastore paths captured:")
for entry in manifest["datastore_paths"]:
    local = BACKUP_DIR / entry["local"]
    n = sum(1 for _ in local.rglob("*") if _.is_file()) if local.exists() else 0
    print(f"  {entry['path']:45s} {n:>6d} files")

print("\nMetadata captured:")
for entry in manifest["metadata_files"]:
    present = (BACKUP_DIR / entry["local"]).exists()
    print(f"  {entry['path']:45s} {'ok' if present else 'MISSING'}")


Created:    2026-08-11T02:58:21.411025+00:00
Workspace:  mlw-llmdatarescue-uksouth-01
Datastore:  large_datastore
Images:     excluded

Datastore paths captured:
  Daily_rainfall_sample/transcriptions             322 files
  Daily_rainfall_sample/outputs/checkpoints       1030 files
  Daily_rainfall_sample/outputs/extractions     3158740 files
  Daily_rainfall_sample/outputs/eval                 0 files
  consensus_data                                  1927 files
  test_data                                        352 files

Metadata captured:
  outputs/model_registry.json                   ok
  outputs/extraction_registry.json              ok
  azureml/config.env                            ok
  azureml/conda-a100.yml                        ok
  azureml/conda.yml                             ok
  azureml/environment-a100.yml                  ok
  azureml/environment.yml                       ok
  azureml/evaluate_job.yml                      ok
  azureml/extract_job.yml                  

## 4. Tear down and spin up (external)

With a verified bundle in hand, the workspace can be safely shut down:

1. **Tear down** the workspace using your existing spin-down mechanism. This destroys the workspace, its datastore, and the underlying storage account.
2. **Spin up** a fresh bare workspace using your existing spin-up mechanism. This must recreate the workspace, the `large_datastore` datastore, and the compute cluster (e.g. `A100x8`) so that the restore step has somewhere to upload to.

These steps are external to this repository, so there is nothing to run here — proceed once the new bare workspace exists and `azureml/config.env` points at it.


## 5. Preview the restore (dry run)

Back on a fresh bare workspace, preview what the restore will do: re-register the environments and re-upload every path recorded in the manifest. `--dry-run` prints the commands without making any changes.


In [ ]:
run("aml_restore.sh", "--from", BACKUP_DIR, "--dry-run")


## 6. Run the restore

Re-register the environments (v100 + a100) and re-upload the datastore data from the bundle. Images are re-uploaded only if they were captured in the backup (or you add `--include-images`). Add `--restore-metadata` to also copy the backed-up registries and `config.env` back into the repository.


In [ ]:
# Real restore into the fresh bare workspace.
run("aml_restore.sh", "--from", BACKUP_DIR)


## 7. Verify the restored workspace

Confirm the new workspace is ready to run jobs:

- **Environments** built successfully (they register asynchronously with `--no-wait`):

  ```bash
  az ml environment show --name weather-doc-extractor \
      --workspace-name "$AML_WORKSPACE" \
      --resource-group "$AML_RESOURCE_GROUP" \
      --subscription "$AML_SUBSCRIPTION"
  ```

- **Checkpoints** are present on the datastore for every registry entry:

  ```bash
  python scripts/check_azure_checkpoints.py
  ```

- **A smoke test** submits and completes — e.g. a small extraction:

  ```bash
  bash scripts/aml_submit.sh --dataset fake --model smolvlm --limit 2 extract
  ```

Once these pass, the workspace is fully repopulated and the backup bundle can be retained as an archive or removed.
